# 02 — Model Experiments (Local)

Reimplements `databricks/notebooks/training/05_train_price_model.py` locally.
Compares XGBoost against simpler alternatives, sweeps hyperparameters, plots
learning curves, and inspects feature importances — all without a Databricks cluster.

**Tracked with local MLflow** — run `uv run mlflow ui` to view results.

In [ ]:
import sys
import os

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
SRC_ROOT  = os.path.join(REPO_ROOT, "databricks")
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge

from src.evaluation.metrics import compute_metrics, passes_quality_gate
from src.features.definitions import ALL_FEATURE_COLS, TARGET_COL
from src.features.validation import check_minimum_rows, check_required_columns
from src.models.hyperparams import DEFAULT_PARAMS, SEARCH_SPACE
from src.models.price_optimizer import PriceOptimizer

# Local MLflow tracking — run `uv run mlflow ui` in local/ to view
mlflow.set_tracking_uri(os.path.join(os.getcwd(), "../mlruns"))
mlflow.set_experiment("pricesense-local-experiments")

DATA_DIR = os.path.join(os.getcwd(), "../data")
print("Setup complete")

## Load and prepare features

In [ ]:
MIN_HISTORY_DAYS = 7
TEST_SIZE        = 0.2

raw = pd.read_csv(os.path.join(DATA_DIR, "sample_gold_features.csv"), parse_dates=["scrape_date"])
raw = raw.dropna(subset=[TARGET_COL, "current_price", "comp_median_price"])

asin_counts = raw.groupby("asin")["scrape_date"].count()
valid_asins = asin_counts[asin_counts >= MIN_HISTORY_DAYS].index
df = raw[raw["asin"].isin(valid_asins)].copy().sort_values(["asin", "scrape_date"])

df["day_of_week"] = df["scrape_date"].dt.dayofweek
df["price_to_median_ratio"] = df["current_price"] / df["comp_median_price"].replace(0, np.nan)

for col in ALL_FEATURE_COLS:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

df_sorted = df.sort_values("scrape_date")
split_idx = int(len(df_sorted) * (1 - TEST_SIZE))
X = df_sorted[ALL_FEATURE_COLS].astype(float)
y = df_sorted[TARGET_COL].astype(float)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Total rows   : {len(df)}")
print(f"Train / Test : {len(X_train)} / {len(X_test)}")
print(f"Features     : {ALL_FEATURE_COLS}")

## Naive baseline — predict comp_median_price directly

In [ ]:
baseline_preds   = df_sorted["comp_median_price"].iloc[split_idx:].fillna(df["comp_median_price"].median()).values
baseline_metrics = compute_metrics(y_test, baseline_preds)
print("Naive baseline (comp_median_price):")
print(f"  RMSE: {baseline_metrics['rmse']:.4f}  MAE: {baseline_metrics['mae']:.4f}  MAPE: {baseline_metrics['mape']:.2f}%")

## Model comparison: Ridge vs RandomForest vs XGBoost

In [ ]:
models = {
    "Naive baseline":  None,
    "Ridge":           Ridge(alpha=1.0),
    "Random Forest":   RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42),
    "XGBoost":         PriceOptimizer(DEFAULT_PARAMS),
}

results = [{"model": "Naive baseline", **baseline_metrics}]

for name, model in models.items():
    if model is None:
        continue
    with mlflow.start_run(run_name=name):
        if isinstance(model, PriceOptimizer):
            model.fit(X_train, y_train, X_val=X_test, y_val=y_test)
            preds = model.predict(X_test)
        else:
            model.fit(X_train, y_train)
            preds = model.predict(X_test)

        m = compute_metrics(y_test, preds)
        mlflow.log_metrics(m)
        results.append({"model": name, **m})

results_df = pd.DataFrame(results).sort_values("rmse")
print(results_df.round(4).to_string(index=False))

## Hyperparameter sensitivity

In [ ]:
sweep_results = []
for n in SEARCH_SPACE["n_estimators"]:
    for d in SEARCH_SPACE["max_depth"]:
        p = {**DEFAULT_PARAMS, "n_estimators": n, "max_depth": d}
        preds = PriceOptimizer(p).fit(X_train, y_train).predict(X_test)
        m = compute_metrics(y_test, preds)
        sweep_results.append({"n_estimators": n, "max_depth": d, **m})

sweep_df = pd.DataFrame(sweep_results).sort_values("rmse")
print("Top 5 hyperparameter combinations:")
print(sweep_df.head(5).round(4).to_string(index=False))
best = sweep_df.iloc[0]
default = sweep_df[(sweep_df["n_estimators"] == DEFAULT_PARAMS["n_estimators"]) & (sweep_df["max_depth"] == DEFAULT_PARAMS["max_depth"])].iloc[0]
print(f"\nDefault RMSE : {default['rmse']:.4f}")
print(f"Best RMSE    : {best['rmse']:.4f}  (n={int(best['n_estimators'])}, depth={int(best['max_depth'])})")

## Learning curve — how much data is enough?

In [ ]:
fracs = np.linspace(0.2, 1.0, 6)
train_rmses, test_rmses, sizes = [], [], []

for frac in fracs:
    n = max(2, int(len(X_train) * frac))
    opt = PriceOptimizer(DEFAULT_PARAMS).fit(X_train.iloc[:n], y_train.iloc[:n])
    train_rmses.append(compute_metrics(y_train.iloc[:n], opt.predict(X_train.iloc[:n]))["rmse"])
    test_rmses.append(compute_metrics(y_test, opt.predict(X_test))["rmse"])
    sizes.append(n)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sizes, train_rmses, marker="o", label="Train RMSE")
ax.plot(sizes, test_rmses,  marker="s", label="Test RMSE")
ax.set_xlabel("Training set size")
ax.set_ylabel("RMSE")
ax.set_title("Learning Curve")
ax.legend()
plt.tight_layout()
plt.show()
print("Flat test RMSE = data-efficient. Still falling = need more scrape history.")

## Feature importances

In [ ]:
final = PriceOptimizer(DEFAULT_PARAMS).fit(X_train, y_train, X_val=X_test, y_val=y_test)
importances = final.feature_importances(ALL_FEATURE_COLS)

fig, ax = plt.subplots(figsize=(8, 5))
importances.sort_values().plot(kind="barh", ax=ax)
ax.set_title("XGBoost Feature Importances")
ax.set_xlabel("Importance score")
plt.tight_layout()
plt.show()

## Residual analysis

In [ ]:
preds = final.predict(X_test)
residuals = pd.DataFrame({
    "actual":        y_test.values,
    "predicted":     preds,
    "residual":      y_test.values - preds,
    "current_price": df_sorted["current_price"].iloc[split_idx:].values,
})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(residuals["actual"], residuals["predicted"], alpha=0.5)
lims = [residuals[["actual", "predicted"]].min().min(), residuals[["actual", "predicted"]].max().max()]
axes[0].plot(lims, lims, "r--")
axes[0].set_xlabel("Actual")
axes[0].set_ylabel("Predicted")
axes[0].set_title("Actual vs Predicted")

axes[1].scatter(residuals["current_price"], residuals["residual"], alpha=0.5)
axes[1].axhline(0, color="r", linestyle="--")
axes[1].set_xlabel("Current price")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residuals vs Price Level")

plt.tight_layout()
plt.show()

final_metrics = compute_metrics(y_test, preds)
passed, msg = passes_quality_gate(final_metrics, max_rmse=10.0, max_mape=15.0)
print(f"Quality gate: {'✓ PASSED' if passed else '✗ FAILED'} — {msg}")